# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jh-emon002/flyrank-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
%pip -q install duckdb huggingface_hub

import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN not found."

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet'"
    f")"
)

FEATURE_START = "2026-03-01"
FEATURE_END = "2026-03-15"

EARLY_START = "2026-03-01"
EARLY_END = "2026-03-07"

RECENT_START = "2026-03-09"
RECENT_END = "2026-03-15"

OUTCOME_START = "2026-03-17"
OUTCOME_END = "2026-03-31"

MIN_IMPRESSIONS = 100
DECLINE_THRESHOLD = 0.80

print("Setup complete.")

Setup complete.


## 1. Two signal checks and my rule
Before fixing the baseline rule, I test two signals that could reasonably
support it. I use only information available before the March 16 decision
point. The outcome used to audit the signals comes from March 17–31.

In [11]:
df = con.sql(f"""
WITH page_windows AS (

    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(DISTINCT CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN report_date
        END) AS feature_days_available,

        COUNT(DISTINCT CASE
            WHEN report_date BETWEEN DATE '{OUTCOME_START}'
                                 AND DATE '{OUTCOME_END}'
             AND gsc_data_available IS TRUE
            THEN report_date
        END) AS outcome_days_available,

        -- Whole safe feature window
        SUM(CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_pre15,

        -- Earlier 7 days
        SUM(CASE
            WHEN report_date BETWEEN DATE '{EARLY_START}'
                                 AND DATE '{EARLY_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_early7,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{EARLY_START}'
                                 AND DATE '{EARLY_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_clicks
            ELSE 0
        END) AS clicks_early7,

        -- Most recent 7 safe days
        SUM(CASE
            WHEN report_date BETWEEN DATE '{RECENT_START}'
                                 AND DATE '{RECENT_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_recent7,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{RECENT_START}'
                                 AND DATE '{RECENT_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_clicks
            ELSE 0
        END) AS clicks_recent7,

        -- Recent weighted average position, review context only
        SUM(CASE
            WHEN report_date BETWEEN DATE '{RECENT_START}'
                                 AND DATE '{RECENT_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_avg_position * gsc_impressions
            ELSE 0
        END)
        /
        NULLIF(
            SUM(CASE
                WHEN report_date BETWEEN DATE '{RECENT_START}'
                                     AND DATE '{RECENT_END}'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END),
            0
        ) AS avg_position_recent7,

        -- FUTURE: evaluation only
        SUM(CASE
            WHEN report_date BETWEEN DATE '{OUTCOME_START}'
                                 AND DATE '{OUTCOME_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_next15

    FROM {MARCH}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM page_windows
WHERE
    feature_days_available = 15
    AND outcome_days_available = 15
    AND impressions_pre15 >= {MIN_IMPRESSIONS}
""").df()

# Safe pre-decision signal
early_denominator = df["impressions_early7"].replace(0, np.nan)

df["recent_trend_pct"] = (
    100.0
    * (df["impressions_recent7"] - df["impressions_early7"])
    / early_denominator
)

# Helpful review context
df["ctr_recent7_pct"] = (
    100.0
    * df["clicks_recent7"]
    / df["impressions_recent7"].replace(0, np.nan)
)

# FUTURE EVALUATION ONLY
df["decline_ratio"] = (
    df["impressions_next15"]
    / df["impressions_pre15"]
)

df["is_declining_next15d"] = (
    df["decline_ratio"] < DECLINE_THRESHOLD
).astype(int)

print("Eligible pages:", len(df))
print(
    "Future decline base rate:",
    round(df["is_declining_next15d"].mean(), 3)
)

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible pages: 58097
Future decline base rate: 0.345


,client_hash_id,content_hash_id,feature_days_available,outcome_days_available,impressions_pre15,impressions_early7,clicks_early7,impressions_recent7,clicks_recent7,avg_position_recent7,impressions_next15,recent_trend_pct,ctr_recent7_pct,decline_ratio,is_declining_next15d
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,15,15,219.0,131.0,1.0,75.0,0.0,2.493333,673.0,-42.748092,0.0,3.073059,0
1,client_62f4a7e64f5e0096,content_d49a012dcb924e31,15,15,246.0,134.0,0.0,98.0,0.0,4.826531,73.0,-26.865672,0.0,0.296748,1
2,client_62f4a7e64f5e0096,content_614baf2af4330bd7,15,15,413.0,203.0,1.0,181.0,0.0,4.110497,329.0,-10.837438,0.0,0.796610,1
3,client_62f4a7e64f5e0096,content_225dc9235023be5f,15,15,279.0,136.0,1.0,124.0,0.0,10.733871,186.0,-8.823529,0.0,0.666667,1
4,client_62f4a7e64f5e0096,content_26f5092ee7f70d45,15,15,2553.0,1333.0,0.0,1019.0,0.0,7.504416,1699.0,-23.555889,0.0,0.665492,1


### Signal 1 — Recent impression volume

**Idea:** Pages with more current search visibility have more business value if
they are at risk. Volume is also the mechanism behind FlyRank's quick-win logic,
so this satisfies the flag-linked-signal requirement.

I bucket recent 7-day impressions and compare each bucket with the observed
next-15-day decline rate. The future label is used only to audit the signal,
never to build the rule.

In [12]:
volume_df = df.copy()

volume_df["volume_bucket"] = pd.cut(
    volume_df["impressions_recent7"],
    bins=[-1, 0, 49, 199, 999, np.inf],
    labels=[
        "0",
        "1-49",
        "50-199",
        "200-999",
        "1000+"
    ]
)

volume_table = (
    volume_df
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("is_declining_next15d", "size"),
        median_recent_impressions=("impressions_recent7", "median"),
        decline_rate=("is_declining_next15d", "mean")
    )
    .reset_index()
)

volume_table["decline_rate_pct"] = (
    100 * volume_table["decline_rate"]
).round(1)

display(
    volume_table[
        [
            "volume_bucket",
            "n",
            "median_recent_impressions",
            "decline_rate_pct"
        ]
    ]
)

,volume_bucket,n,median_recent_impressions,decline_rate_pct
0,1-49,1572,43.0,36.1
1,50-199,19773,107.0,35.2
2,200-999,24506,407.0,33.6
3,1000+,12246,1916.0,34.9


**Verdict: MIXED**

The observed future-decline rate does not move monotonically with current
impression volume. I therefore do not treat volume as a predictor of decline.
I retain it only as an impact/prioritisation signal: among otherwise similar
declining pages, the page still receiving more impressions is more valuable
to inspect first.

### Signal 2 — Recent impression trend

**Idea:** A page that has already started losing impressions during the safe
feature window may be more likely to remain weak in the following 15 days.

I compare March 1–7 with March 9–15. Both periods occur before the March 16
decision point.

In [13]:
trend_df = df[df["recent_trend_pct"].notna()].copy()

trend_df["trend_bucket"] = pd.cut(
    trend_df["recent_trend_pct"],
    bins=[-np.inf, -50, -20, 20, 50, np.inf],
    labels=[
        "strong_down",
        "down",
        "stable",
        "up",
        "strong_up"
    ],
    include_lowest=True
)

trend_table = (
    trend_df
    .groupby("trend_bucket", observed=True)
    .agg(
        n=("is_declining_next15d", "size"),
        median_trend_pct=("recent_trend_pct", "median"),
        decline_rate=("is_declining_next15d", "mean")
    )
    .reset_index()
)

trend_table["median_trend_pct"] = (
    trend_table["median_trend_pct"].round(1)
)

trend_table["decline_rate_pct"] = (
    100 * trend_table["decline_rate"]
).round(1)

display(
    trend_table[
        [
            "trend_bucket",
            "n",
            "median_trend_pct",
            "decline_rate_pct"
        ]
    ]
)

,trend_bucket,n,median_trend_pct,decline_rate_pct
0,strong_down,4644,-59.5,60.8
1,down,16113,-32.9,40.7
2,stable,23559,-2.7,30.5
3,up,7186,31.9,26.8
4,strong_up,6595,88.7,23.5


**Verdict: CONFIRMED**

Pages in the down and strong-down buckets show a higher observed future-decline
rate than stable or growing pages. The pattern is directional rather than causal,
but it supports using recent impression decline in the baseline rule.

### My baseline rule

A page enters the review rule when:

1. it still has at least one impression in the recent seven-day window; and
2. recent seven-day impressions are at least 20% below the earlier seven-day window.

I rank triggered pages using:

`log(1 + recent impressions) × decline severity`

The logarithm prevents extremely large pages from dominating the queue purely
because of scale. This is a transparent prioritisation rule, not a fitted model.

**Reason code:** `recent_decline_still_visible`

**Action:** `review_for_refresh`

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
queue = df.copy()

RULE_DECLINE_THRESHOLD = -20.0

# Severity is 0 for growth, up to 1 for a complete 100% decline.
queue["decline_strength"] = (
    -queue["recent_trend_pct"] / 100.0
).clip(lower=0, upper=1).fillna(0)

queue["rule_trigger"] = (
    (queue["recent_trend_pct"] <= RULE_DECLINE_THRESHOLD)
    & (queue["impressions_recent7"] > 0)
)

queue["baseline_score"] = np.where(
    queue["rule_trigger"],
    np.log1p(queue["impressions_recent7"])
    * queue["decline_strength"],
    0.0
)

# Exactly one reason code per row.
queue["reason_code"] = np.where(
    queue["rule_trigger"],
    "recent_decline_still_visible",
    "none"
)

queue["action"] = np.where(
    queue["rule_trigger"],
    "review_for_refresh",
    "monitor"
)

queue = (
    queue
    .sort_values(
        ["baseline_score", "impressions_recent7"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

print("Rows in full ranked queue:", len(queue))
print("Pages triggered for review:", int(queue["rule_trigger"].sum()))

display(
    queue[
        [
            "rank",
            "content_hash_id",
            "baseline_score",
            "impressions_recent7",
            "recent_trend_pct",
            "reason_code",
            "action"
        ]
    ].head(20)
)


Rows in full ranked queue: 58097
Pages triggered for review: 20757


,rank,content_hash_id,baseline_score,impressions_recent7,recent_trend_pct,reason_code,action
0,1,content_34a70fea29d15f24,7.803010,9271.0,-85.421122,recent_decline_still_visible,review_for_refresh
1,2,content_945d6ff91386c817,7.412858,6758.0,-84.059065,recent_decline_still_visible,review_for_refresh
2,3,content_0c5606abaaab3178,6.999412,3361.0,-86.196558,recent_decline_still_visible,review_for_refresh
3,4,content_ed50f7f4237a3d02,6.903974,6011.0,-79.342223,recent_decline_still_visible,review_for_refresh
4,5,content_6a9c79f55413b447,6.758071,7998.0,-75.197693,recent_decline_still_visible,review_for_refresh
5,6,content_0bca6d9a85a9b408,6.472725,1085.0,-92.596383,recent_decline_still_visible,review_for_refresh
6,7,content_8abf2671c081e29e,6.275614,860.0,-92.860701,recent_decline_still_visible,review_for_refresh
7,8,content_57487b3ef3d84b7d,6.260007,2614.0,-79.552566,recent_decline_still_visible,review_for_refresh
8,9,content_8903bf34506a2f39,6.235748,1512.0,-85.166291,recent_decline_still_visible,review_for_refresh
9,10,content_1ff6231687184dec,6.230641,795.0,-93.278661,recent_decline_still_visible,review_for_refresh


In [15]:
TARGET = "is_declining_next15d"

base_rate = queue[TARGET].mean()

def precision_at_k(frame, k):
    return frame.head(k)[TARGET].mean()

print(f"Base decline rate: {base_rate:.3f}")

for k in [10, 20, 50, 100]:
    print(
        f"Precision@{k}: "
        f"{precision_at_k(queue, k):.3f}"
    )

Base decline rate: 0.345
Precision@10: 1.000
Precision@20: 0.850
Precision@50: 0.760
Precision@100: 0.700


In [17]:
OUTPUT_PATH = Path(
    "work/outputs/baseline_action_score.csv"
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "impressions_recent7",
    "recent_trend_pct",
    "reason_code",
    "action"
]

queue[output_columns].to_csv(
    OUTPUT_PATH,
    index=False
)

print("Wrote:", OUTPUT_PATH)
print("Absolute location:", OUTPUT_PATH.resolve())

Wrote: work/outputs/baseline_action_score.csv
Absolute location: /content/work/outputs/baseline_action_score.csv


## 3. Top-10 review

*For each of the top 10: action, reason code, and what would make it wrong.*

In [18]:
top10 = (
    queue[queue["action"] == "review_for_refresh"]
    .head(10)
    .copy()
)

review_columns = [
    "rank",
    "content_hash_id",
    "baseline_score",
    "impressions_early7",
    "impressions_recent7",
    "recent_trend_pct",
    "ctr_recent7_pct",
    "avg_position_recent7",
    "is_declining_next15d"
]

display(top10[review_columns])

,rank,content_hash_id,baseline_score,impressions_early7,impressions_recent7,recent_trend_pct,ctr_recent7_pct,avg_position_recent7,is_declining_next15d
0,1,content_34a70fea29d15f24,7.803010,63592.0,9271.0,-85.421122,0.053932,3.728832,1
1,2,content_945d6ff91386c817,7.412858,42394.0,6758.0,-84.059065,0.029595,7.185706,1
2,3,content_0c5606abaaab3178,6.999412,24349.0,3361.0,-86.196558,0.000000,6.535257,1
3,4,content_ed50f7f4237a3d02,6.903974,29098.0,6011.0,-79.342223,0.282815,2.057561,1
4,5,content_6a9c79f55413b447,6.758071,32247.0,7998.0,-75.197693,0.287572,2.621405,1
5,6,content_0bca6d9a85a9b408,6.472725,14655.0,1085.0,-92.596383,0.184332,4.552995,1
6,7,content_8abf2671c081e29e,6.275614,12046.0,860.0,-92.860701,0.232558,3.336047,1
7,8,content_57487b3ef3d84b7d,6.260007,12784.0,2614.0,-79.552566,0.076511,39.164116,1
8,9,content_8903bf34506a2f39,6.235748,10193.0,1512.0,-85.166291,0.462963,6.166005,1
9,10,content_1ff6231687184dec,6.230641,11828.0,795.0,-93.278661,0.000000,8.854088,1


| Rank | Action             | Reason                                                                                                                             | What could make it wrong                                                                                                                                                        |
| ---: | ------------------ | ---------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
|    1 | Review for refresh | Impressions fell from **63,592 to 9,271 (-85.42%)**, while the page still retains substantial recent visibility                    | The decline could be caused by a temporary indexing/tracking issue or a broad drop in search demand rather than outdated content                                                |
|    2 | Review for refresh | Impressions fell from **42,394 to 6,758 (-84.06%)**, with meaningful search visibility still remaining                             | A SERP or ranking change outside the page itself may be responsible, so refreshing the content may not recover traffic                                                          |
|    3 | Review for refresh | Impressions fell from **24,349 to 3,361 (-86.20%)**, indicating a severe decline while some search demand remains                  | The topic may be seasonal or demand may have fallen independently of content quality                                                                                            |
|    4 | Review for refresh | Impressions fell from **29,098 to 6,011 (-79.34%)**, and the page still has relatively high recent exposure                        | The decline may reflect a temporary change in search demand rather than a problem that requires a content refresh                                                               |
|    5 | Review for refresh | Impressions fell from **32,247 to 7,998 (-75.20%)**, leaving one of the highest recent impression volumes among the top candidates | Because the decline is less severe than several other candidates, the score may be influenced strongly by its high remaining volume                                             |
|    6 | Review for refresh | Impressions fell from **14,655 to 1,085 (-92.60%)**, showing a near-total loss of previous visibility                              | The sharp fall could be caused by de-indexing, a technical SEO issue, or a temporary SERP change rather than stale content                                                      |
|    7 | Review for refresh | Impressions fell from **12,046 to 860 (-92.86%)**, indicating an extremely severe deterioration                                    | The page may have been intentionally retired, redirected, or affected by a technical/indexing issue that a content refresh would not solve                                      |
|    8 | Review for refresh | Impressions fell from **12,784 to 2,614 (-79.55%)**, so the page still receives meaningful impressions despite a large decline     | Its recent average position is around **39.2**, so poor ranking or weak search relevance may be the main problem rather than content freshness                                  |
|    9 | Review for refresh | Impressions fell from **10,193 to 1,512 (-85.17%)**, showing a strong decline while some recent visibility remains                 | The decline could be due to changing search demand or SERP competition rather than deterioration of the page itself                                                             |
|   10 | Review for refresh | Impressions fell from **11,828 to 795 (-93.28%)**, making it one of the strongest declines in the queue                            | With only 795 recent impressions remaining, the potential impact may be limited, and the decline could reflect technical or indexing problems rather than a refresh opportunity |


| Rank | Action             | Reason                                         | Confidence note                                                                     | What could make it wrong                                                             |
| ---: | ------------------ | ---------------------------------------------- | ----------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------ |
|    1 | Review for refresh | 83,772 impressions and a 99.95% recent decline | Very high — extreme deterioration plus very high exposure                           | Temporary tracking/indexing issue or short-lived demand shock                        |
|    2 | Review for refresh | 73,639 impressions and an 85.37% decline       | High — very large exposure, though decline is less severe than many other top picks | Volume weighting may be lifting it above pages with stronger deterioration           |
|    3 | Review for refresh | 12,020 impressions and a 97.20% decline        | Very high — near-total recent collapse                                              | Seasonal demand or temporary SERP/indexing change                                    |
|    4 | Review for refresh | 49,314 impressions and an 84.01% decline       | High — major exposure at risk                                                       | High volume may be contributing more to rank than decline severity                   |
|    5 | Review for refresh | 15,965 impressions and a 92.58% decline        | Very high — severe decline with meaningful demand                                   | Search demand for the topic may have fallen independently of content quality         |
|    6 | Review for refresh | 14,244 impressions and a 93.54% decline        | Very high                                                                           | Temporary technical/search visibility issue rather than refresh need                 |
|    7 | Review for refresh | 27,715 impressions and an 86.19% decline       | High                                                                                | External demand or SERP changes may explain the loss                                 |
|    8 | Review for refresh | 12,634 impressions and a 93.28% decline        | Very high                                                                           | Recent decline could be temporary rather than persistent                             |
|    9 | Review for refresh | 9,307 impressions and a 94.11% decline         | Very high                                                                           | Topic seasonality or indexing instability                                            |
|   10 | Review for refresh | 5,248 impressions and a 99.94% decline         | Very high on decline, moderate on opportunity                                       | Near-total fall could reflect tracking/indexing failure rather than content weakness |
|   11 | Review for refresh | 4,962 impressions and a 99.90% decline         | Very high on decline                                                                | Same: extreme collapse may be technical rather than editorial                        |
|   12 | Review for refresh | 9,103 impressions and a 92.81% decline         | Very high                                                                           | Search-demand shift could make refresh ineffective                                   |
|   13 | Review for refresh | 19,811 impressions and an 85.49% decline       | High                                                                                | Volume weighting may be responsible for its high rank                                |
|   14 | Review for refresh | 18,917 impressions and an 85.80% decline       | High                                                                                | Similar risk: strong volume but less severe decline than many lower-volume pages     |
|   15 | Review for refresh | 36,945 impressions and an 80.28% decline       | High opportunity, slightly lower risk confidence                                    | High exposure strongly boosts score despite the mildest decline in the top 20        |
|   16 | Review for refresh | 7,568 impressions and a 93.13% decline         | Very high                                                                           | Could reflect temporary demand or SERP volatility                                    |
|   17 | Review for refresh | 8,975 impressions and a 90.96% decline         | Very high                                                                           | Decline may not be caused by content freshness                                       |
|   18 | Review for refresh | 6,600 impressions and a 94.13% decline         | Very high                                                                           | Short-term disruption may reverse without intervention                               |
|   19 | Review for refresh | 5,940 impressions and a 94.79% decline         | Very high                                                                           | Could be seasonal or technical rather than editorial                                 |
|   20 | Review for refresh | 4,266 impressions and a 97.43% decline         | Very high on decline, lower opportunity                                             | Severe decline but smaller exposure than most top-ranked pages                       |


## 4. Weak picks + leakage check

Rank 2, 4, 15 are weak picks. Because volume was only MIXED as a future-decline signal, and these rows have somewhat milder decline than many others but get pushed upward by huge impression volume.

In [19]:
# Look for false positives among the highest-ranked review candidates.
weak_picks = (
    queue[
        (queue["action"] == "review_for_refresh")
        & (queue["is_declining_next15d"] == 0)
    ]
    .head(5)
)

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "baseline_score",
            "impressions_early7",
            "impressions_recent7",
            "recent_trend_pct",
            "ctr_recent7_pct",
            "avg_position_recent7",
            "is_declining_next15d"
        ]
    ]
)

,rank,content_hash_id,baseline_score,impressions_early7,impressions_recent7,recent_trend_pct,ctr_recent7_pct,avg_position_recent7,is_declining_next15d
10,11,content_2f6c3048d8e75f26,6.215555,11104.0,2059.0,-81.457133,0.000000,5.773677,0
16,17,content_f0f6a030c428f851,6.040057,8789.0,1585.0,-81.966094,0.315457,4.214511,0
17,18,content_252aa5480bb1f8d7,6.031038,23019.0,7448.0,-67.644120,0.080559,2.276719,0
22,23,content_c7064a001663e57b,5.933188,11619.0,3014.0,-74.059730,0.597213,4.845388,0
24,25,content_e7b5dd4dff461ad2,5.926837,60220.0,24971.0,-58.533710,1.738016,4.647191,0


**Leakage check:** PASS.

The baseline score uses only `impressions_recent7` and `recent_trend_pct`,
both computed before the March 16 decision point. `impressions_next15`,
`decline_ratio`, and `is_declining_next15d` are used only for retrospective
signal checks and Precision@K evaluation. No FlyRank product flag,
future-window value, label, or label-derived field enters the score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.